In [20]:
# ============================================================
#  Project Name: brandalyze
#  File: tweet_model_training.ipynb
#  Author: dngi (https://twitter.com/_dngi)
#  Created: <2026-02-13>
#  
#  Copyright (c) 2026 Dom G. (https://twitter.com/_dngi)
#  All rights reserved.
#
#  This notebook and its contents are confidential and
#  proprietary. Unauthorized copying, distribution,
#  modification, or use is strictly prohibited.
# ============================================================

# Brandalyze Tweet Model Training

## Setup and Data Loading
- loading the labeled dataset and prepping for ML training

In [21]:
import json
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# load data
with open('labeled-viral-tweets.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

df = pd.DataFrame(data)

# filter only labeled tweets
df = df[df['labels'].notna()].copy()

# check gpu availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"cuDNN version: {torch.backends.cudnn.version()}")

Using device: cuda
GPU: NVIDIA GeForce RTX 3060
Memory: 12.00 GB
PyTorch version: 2.7.1+cu118
CUDA available: True
CUDA version: 11.8
cuDNN version: 90100


## Prepare Training Data
- Extract features and labels for each classification task
- clean data by removing any `unknown` labels

In [22]:
# extract labels
df['format'] = df['labels'].apply(lambda x: x.get('format'))
df['hookQuality'] = df['labels'].apply(lambda x: x.get('hookQuality'))
df['closerType'] = df['labels'].apply(lambda x: x.get('closerType'))

# prepare text input
df['text_clean'] = df['text'].str.strip()

df = df[df['format'] != 'unknown']
df = df[df['hookQuality'] != 'unknown']
df = df[df['closerType'] != 'unknown']

print(f"total training samples: {len(df)}")

total training samples: 6987


## Train/Val/Test Split
- Split data using stratification to maintain class balance

In [23]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    random_state=42,
    stratify=df['format']
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df['format']
)

print(f"Train: {len(train_df)} ({len(train_df) / len(df)*100:.1f}%)")
print(f"Val: {len(val_df)} ({len(val_df) / len(df)*100:.1f}%)")
print(f"Test: {len(test_df)} ({len(test_df) / len(df)*100:.1f}%)")

Train: 4890 (70.0%)
Val: 1048 (15.0%)
Test: 1049 (15.0%)


## Model 1: Format Classifier

In [24]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# tokenize
train_encodings = tokenizer(
    train_df['text_clean'].tolist(),
    truncation=True,
    padding=True,
    max_length=280
)

val_encodings = tokenizer(
    val_df['text_clean'].tolist(),
    truncation=True,
    padding=True,
    max_length=280
)

test_encodings = tokenizer(
    test_df['text_clean'].tolist(),
    truncation=True,
    padding=True,
    max_length=280
)

label_encoder = LabelEncoder()
train_labels = label_encoder.fit_transform(train_df['format'])
val_labels = label_encoder.transform(val_df['format'])
test_labels = label_encoder.transform(test_df['format'])

# create a pytorch dataset
import torch

class TweetDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    
    def __len__(self):
        return len(self.labels)
    
train_dataset = TweetDataset(train_encodings, train_labels)
val_dataset = TweetDataset(val_encodings, val_labels)
test_dataset = TweetDataset(test_encodings, test_labels)


## Training Config
- setting up training params

In [25]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir='./models/format_classifier',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    use_cpu=False
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_encoder.classes_)
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 780.96it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your down

## Train and Evaluate

- run training and check metrics

In [26]:
trainer.train()

# evaluate on the test set
predictions = trainer.predict(test_dataset)
pred_labels = np.argmax(predictions.predictions, axis=1)

from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(
    test_labels,
    pred_labels,
    target_names=label_encoder.classes_
))

# save model
model.save_pretrained('./models/format_classifier_final')
tokenizer.save_pretrained('./models/format_classifier_final')

# save label encoder
import joblib
joblib.dump(label_encoder, './models/format_label_encoder.pkl')

Epoch,Training Loss,Validation Loss
1,1.484742,1.500023
2,1.135080,1.228173
3,0.630747,1.281707


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.23it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


               precision    recall  f1-score   support

  educational       0.25      0.04      0.06        28
 entertaining       0.63      0.32      0.42       104
  informative       0.60      0.65      0.62       313
inspirational       0.71      0.11      0.19        47
         news       0.79      0.48      0.60        31
      opinion       0.40      0.67      0.50       183
     question       0.50      0.76      0.60        34
     shitpost       0.61      0.55      0.58       244
        story       0.76      0.58      0.66        65

     accuracy                           0.55      1049
    macro avg       0.58      0.46      0.47      1049
 weighted avg       0.58      0.55      0.54      1049



Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.41s/it]


['./models/format_label_encoder.pkl']

## Model 2: Hook Quality Classifier

In [27]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# tokenize
train_encodings = tokenizer(
    train_df['text_clean'].tolist(),
    truncation=True,
    padding=True,
    maxLength=280
)

val_encodings = tokenizer(
    val_df['text_clean'].tolist(),
    truncation=True,
    padding=True,
    maxLength=280
)


test_encodings = tokenizer(
    test_df['text_clean'].tolist(),
    truncation=True,
    padding=True,
    maxLength=280
)

label_encoder = LabelEncoder()
train_labels = label_encoder.fit_transform(train_df['hookQuality'])
val_labels = label_encoder.fit_transform(val_df['hookQuality'])
test_labels = label_encoder.fit_transform(test_df['hookQuality'])

# create a pytorch dataset
import torch

class TweetDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    
    def __len__(self):
        return len(self.labels)
    
train_dataset = TweetDataset(train_encodings, train_labels)
val_dataset = TweetDataset(val_encodings, val_labels)
test_dataset = TweetDataset(test_encodings, test_labels)

## Training Config
- hookQuality classifier training params

In [28]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir='./models/hook_classifier',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    use_cpu=False
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_encoder.classes_)
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 662.13it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your down

## Train and Evaluate
- run training on hookQuality classifier and check metrics

In [29]:
trainer.train()

# evaluate on the test set
predictions = trainer.predict(test_dataset)
pred_labels = np.argmax(predictions.predictions, axis=1)

from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(
    test_labels,
    pred_labels,
    target_names=label_encoder.classes_
))

# save model
model.save_pretrained('./models/hook_classifier_final')
tokenizer.save_pretrained('./models/hook_classifier_final')

import joblib
joblib.dump(label_encoder, './models/hook_label_encoder.pkl')

Epoch,Training Loss,Validation Loss
1,0.875105,0.864919
2,1.030196,0.821431
3,0.614194,0.862508


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.30it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


              precision    recall  f1-score   support

      medium       0.25      0.26      0.25       101
        none       0.79      0.91      0.85       685
      strong       0.49      0.50      0.49       104
        weak       0.29      0.08      0.13       159

    accuracy                           0.68      1049
   macro avg       0.45      0.44      0.43      1049
weighted avg       0.63      0.68      0.64      1049



Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.06s/it]


['./models/hook_label_encoder.pkl']

## Model 3: Closer Type Classifier

In [30]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# tokenize
train_encodings = tokenizer(
    train_df['text_clean'].tolist(),
    truncation=True,
    padding=True,
    maxLength=280
)

val_encodings = tokenizer(
    val_df['text_clean'].tolist(),
    truncation=True,
    padding=True,
    maxLength=280
)

test_encodings = tokenizer(
    test_df['text_clean'].tolist(),
    truncation=True,
    padding=True,
    maxLength=280
)

label_encoder = LabelEncoder()
train_labels = label_encoder.fit_transform(train_df['closerType'])
val_labels = label_encoder.fit_transform(val_df['closerType'])
test_labels = label_encoder.fit_transform(test_df['closerType'])

# create a pytorch
import torch

class TweetDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    
    def __len__(self):
        return len(self.labels)
    
train_dataset = TweetDataset(train_encodings, train_labels)
val_dataset = TweetDataset(val_encodings, val_labels)
test_dataset = TweetDataset(test_encodings, test_labels)

## Training Config
- closerType classifier training params

In [31]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir='./models/closer_classifier',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    use_cpu=False
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_encoder.classes_)
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 724.22it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your down

## Train and Evaluate
- run training on the closerType classifier and check metrics

In [32]:
trainer.train()

# evaluate on the test set
predictions = trainer.predict(test_dataset)
pred_labels = np.argmax(predictions.predictions, axis=1)

from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(
    test_labels,
    pred_labels,
    target_names=label_encoder.classes_
))

# save model
model.save_pretrained('./models/closer_classifier_final')
tokenizer.save_pretrained('./models/closer_classifier_final')

import joblib
joblib.dump(label_encoder, './models/closer_label_encoder.pkl')

Epoch,Training Loss,Validation Loss
1,0.670743,0.693015
2,0.563873,0.589142
3,0.352219,0.590578


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.16it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


d:\projects\brandalyze\apps\backend\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\projects\brandalyze\apps\backend\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\projects\brandalyze\apps\backend\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(r

              precision    recall  f1-score   support

 cliffhanger       0.59      0.50      0.54        52
         cta       0.00      0.00      0.00        17
        none       0.78      0.96      0.86       759
    question       1.00      0.15      0.26        27
   statement       0.59      0.21      0.31       194

    accuracy                           0.76      1049
   macro avg       0.59      0.36      0.39      1049
weighted avg       0.73      0.76      0.72      1049



Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.47it/s]


['./models/closer_label_encoder.pkl']